## rag(retrival augmented generation) from scratch

### ingestion pipeline

In [1]:
# loading text/data from a file(pdf)
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader("../data/", glob="**/*.pdf", loader_cls=PyPDFLoader)
documents = dir_loader.load()
print(documents)

/tmp/ipykernel_41927/976910581.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


[Document(metadata={'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'creator': 'Word', 'creationdate': '2021-07-06T00:34:19+00:00', 'moddate': '2023-03-21T13:45:49+03:00', 'title': '', 'source': '../data/E08-003 - Electrical Engineering Fundamentals - Motors, Generators & Power Distribution - US.pdf', 'total_pages': 85, 'page': 0, 'page_label': '1'}, page_content='Electrical Engineering Fundamentals: \nMotors, Generators & Power Distribution\nCourse No: E08-003\nCredit: 8 PDH\nS. Bobby Rauf, P.E., C.E.M., MBA\nwww.cedengineering.com \nContinuing Education and Development, Inc. \nP: (877) 322-5800 \ninfo@cedengineering.com'), Document(metadata={'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'creator': 'Word', 'creationdate': '2021-07-06T00:34:19+00:00', 'moddate': '2023-03-21T13:45:49+03:00', 'title': '', 'source': '../data/E08-003 - Electrical Engineering Fundamentals - Motors, Generators & Power Distribution - US.pdf', 'total_pages': 85, 'page': 1,

In [2]:
# creating chunks from text loaded

from langchain_text_splitters import RecursiveCharacterTextSplitter


def process_and_chunk(documents, chunk_size=1000, chunk_overlap=500):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks.")
    return split_docs


chunks = process_and_chunk(documents)
# chunks

split 85 documents into 190 chunks.


In [3]:
# embedding
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, PreTrainedModel
import numpy as np
from typing import List, Any, Dict


class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading embedding manager: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"model loaded with embedding dimentions: {self.model.get_embedding_dimension()}"
            )
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("model not loaded!")
        print(f"generating embedding for {len(texts)} chunks..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()
embedding_manager

/home/ayush/Desktop/projects/rag-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading embedding manager: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1864.85it/s]


model loaded with embedding dimentions: 384


In [4]:
# vector store
import os
import chromadb
import hashlib
import numpy as np

class VectorManager:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._load_store()

    def _load_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                   "description": "pdf documents for rag",
                    "hnsw:space": "cosine",
                    },)
            print(f"vector store initialized, collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error loading vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("no of documents does not match no of embedding")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Deterministic ID: hash of source + page + chunk index
            source = doc.metadata.get('source', 'unknown')
            page = doc.metadata.get('page', 0)
            doc_hash = hashlib.md5(doc.page_content.encode()).hexdigest()[:8]
            hash_input = f"{source}_{page}_{doc_hash}".encode()
            doc_id = hashlib.md5(hash_input).hexdigest()[:12]
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.upsert(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"added {len(documents)} documents into vector store")
            print(f"total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error adding documents to vector store: {e}")
            raise

vector_store=VectorManager()
vector_store

vector store initialized, collection: pdf_documents
existing documents in collection: 190


In [5]:
# convert text into embeddings
texts = [doc.page_content for doc in chunks]

# generate embedding
embeddings = embedding_manager.generate_embedding(texts)

# store in vector db
vector_store.add_documents(chunks, embeddings)

generating embedding for 190 chunks..


Batches: 100%|██████████| 6/6 [00:31<00:00,  5.29s/it]


generated embeddings with shape: (190, 384)
added 190 documents into vector store
total documents in collection: 190


### retrieval pipeline

In [6]:
class RagRetriever:
    def __init__(self, vector_store: VectorManager, embedding_manager:EmbeddingManager ):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager 

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, any]]:
        print(f"retrieving documents for query: {query}")
        print(f"top k: {top_k}, score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )
                print(f"retireved {len(retrieved_docs)} documents after search")
            else:
                print("no documents found")
            return retrieved_docs

        except Exception as e:
            print(f"error during retrieval: {e}")


rag_retriever = RagRetriever(vector_store, embedding_manager)
rag_retriever

### vector store to llm output generation

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API"))

True

In [8]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-20b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
        

In [9]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-20b
Groq LLM initialized successfully!


In [10]:
rag_retriever.retrieve("what are synchronous machines")

retrieving documents for query: what are synchronous machines
top k: 5, score threshold: 0.0
generating embedding for 1 chunks..


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 15.75it/s]

generated embeddings with shape: (1, 384)
retireved 5 documents after search


[{'id': 'dbc1d950839b',
  'content': 'Synchronous Motors \nSynchronous motors are AC induction motors that have no slip. In \nother words, in synchronous motors, the speed of rotor or motor shaft is the \nsame as the rotational speed of the stator winding magnetic field. This is \naccomplished through the excitation of the rotor through the field windings \nsuch that the rotor is able to “catch up” to the rotational speed of the stator. \nThis characteristic permits synchronous motors to induce leading currents in \nthe branch circuit. Therefore, synchronous motors can serve as alternative \nmeans for power factor correction; albeit, application of power factor \ncorrecting capacitors is more economical and effective in most lagging power \nfactor situations. \nExample 2.6 \nA three phase induction motor delivers 550 kW at a power factor of 82%. \nDetermine the size of synchronous motor, in kVA, that should be installed to \ncarry a load of 250 hp load and, at the same time, raise the 

In [11]:
# llm integration
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=5):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [21]:
answer = rag_simple("how does dc motor differ from ac motor?", rag_retriever, llm)
print(answer)

retrieving documents for query: how does dc motor differ from ac motor?
top k: 5, score threshold: 0.0
generating embedding for 1 chunks..


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.77it/s]

generated embeddings with shape: (1, 384)
retireved 5 documents after search


**DC motor vs. AC motor**

| Feature | DC motor | AC motor |
|---------|----------|----------|
| **Power source** | Direct current (DC) | Alternating current (AC) |
| **Commutation** | Mechanical commutator & brushes (self‑commutated) | No commutator; commutation is electrical (induction or synchronous) |
| **Speed control** | Directly by changing armature voltage or field current (or via a DC drive) – no inverter needed | Requires an inverter or variable‑frequency drive to vary speed; otherwise speed is fixed by supply frequency |
| **Construction** | Field winding + armature winding, commutator, brushes | Stator winding + rotor winding (or squirrel‑cage) – no brushes |
| **Typical use** | Small toys, disk drives, applications needing precise speed/torque control | Large industrial drives, HVAC, appliances; AC induction motors dominate |
| **Key advantage** | Simple direct‑drive from DC, easy speed variation | Robust, low‑maintenance, no brushes, high efficiency at constant speed |

I